In [6]:
import mesa_reader as mr 
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mesaPlot as mp
import os


In [7]:
#constants

G_const = 6.6743 * pow(10,-8)                 #g^-1 cm^3 s^-2 
M_sun = 1.9891 * pow(10,33)             #g
R_sun = 6.957 * pow(10,10)              #cm
c = 3 * pow(10,10)                      #cm s^-1


#model parameters

M_ns = 1.33 * M_sun                     #g
R_ns = 1000000                          #cm
f0 = 250                                #Hz
omega0 = f0*2*np.pi                     #Hz
e0 = 0.177
p_0 = 1/f0                               #s

eta1 = 1
eta2 = 0.1
alpha_ce = 1
R_eq = 10**6                            #cm
R_const = (8.31 * 10**7) / (6.022 * 10**(23)) #erg/K

#conversions

s_to_yr = 3.17098 * pow(10,-8)
yr_to_s = 3.154 * 10**7
pc_to_cm = 3.086 * pow(10,18)
D = 2.8*1000*pc_to_cm

In [11]:
logs_path = 'C:/Users/abhav/Documents/UIUC_Graduate/Research/CE_GW/Temp_cluster_scp/LOGS/'
logs_data = mr.MesaLogDir(logs_path)
h_data = logs_data.history_data
p_index_data = logs_data.profiles

In [12]:
p1 = mr.MesaData(logs_path+'/profile1.data')
age_array_for_profile = np.array([])
model_array_for_age = np.array([])
profile_array_for_age = np.array([]) #array for ages in integers

for i in range(p_index_data.profile_numbers.max()):
    p_temp = mr.MesaData(logs_path+f'/profile{i+1}.data')
    age_array_for_profile = np.append(age_array_for_profile, p_temp.star_age)
for i in range(np.int64(np.floor(np.max(h_data.data('star_age'))))+1):
    temp = np.where(h_data.data('star_age')>i)[0][0]
    model_array_for_age = np.int64(np.append(model_array_for_age, h_data.data('model_number')[temp]))
    temp = np.where(age_array_for_profile>=i)[0][0]
    profile_array_for_age = np.int64(np.append(profile_array_for_age, temp+1))
model_array_for_age = np.append(model_array_for_age, h_data.data('model_number')[-1])
profile_array_for_age = np.append(profile_array_for_age, p_index_data.profile_numbers.max())

In [13]:
print(profile_array_for_age)

[  1 192]


In [14]:
print('\n'.join(p1.bulk_names))

zone
mass
logR
logT
logRho
logP
x_mass_fraction_H
y_mass_fraction_He
z_mass_fraction_metals
dm
velocity
scale_height
pressure_scale_height
dr
dr_div_cs
acoustic_radius
cell_collapse_time
compression_gradient
energy
entropy
grada
csound
v_div_cs
thermal_time_to_surface
pp
cno
tri_alpha
opacity
luminosity
total_energy
cell_specific_IE
cell_specific_KE
total_energy_sign
mlt_mixing_length
conv_vel
gradT
gradr
tau
omega
v
v_div_vesc
u
u_face
extra_heat
extra_L


In [15]:
print('\n'.join(h_data.bulk_names))


model_number
num_zones
star_age
time_step
log_dt
star_mass
log_xmstar
star_mdot
log_abs_mdot
mass_conv_core
conv_mx1_top
conv_mx1_bot
conv_mx2_top
conv_mx2_bot
mx1_top
mx1_bot
mx2_top
mx2_bot
log_LH
log_LHe
log_LZ
log_Lnuc
extra_L
pp
cno
tri_alpha
epsnuc_M_1
epsnuc_M_2
epsnuc_M_3
epsnuc_M_4
epsnuc_M_5
epsnuc_M_6
epsnuc_M_7
epsnuc_M_8
he_core_mass
co_core_mass
one_core_mass
fe_core_mass
neutron_rich_core_mass
dynamic_timescale
kh_timescale
nuc_timescale
log_Teff
photosphere_black_body_T
luminosity
log_L
log_R
log_g
log_surf_cell_temperature
v_div_csound_surf
log_cntr_P
log_cntr_Rho
log_cntr_T
center_mu
center_ye
center_abar
center_h1
center_he4
center_c12
center_o16
surface_c12
surface_o16
total_mass_h1
total_mass_he4
total_energy
total_gravitational_energy
tot_PE
total_internal_energy
num_retries
num_iters
aorb
Einj_per_dt
M_ns
M_enc
M_acc
rho
Qmax
Qtb
Q
omega
mom_inert
strain
scale_height
csound
lnPgas
lnT
u_k
v_rel
mdot_hl
fd_hl
edot_hl
Ra
mdot_mr15
fd_mr15
edot_mr15
fd
edot


In [16]:
def plot_xy(x,y,ax,xlabel = None, ylabel = None, xmin = None, xmax = None, ymin = None, ymax = None, title = None, ylog = False, xlog = False, legend = None, scatter = False, linestyle = None, marker = None, color = None):
    if not scatter:
        if linestyle:
            ax.plot(x,y, label = legend, linestyle = linestyle, color = color)
        else:
            ax.plot(x,y, label = legend, color = color)
    else:
        if marker:
            ax.scatter(x,y, label = legend, marker = marker, color = color)
        else:
            ax.scatter(x,y, label = legend, color = color)
    if ylog:
        ax.set_yscale('log')
    if xlog:
        ax.set_xscale('log')
    if xlabel:
        ax.set_xlabel(xlabel, fontsize = 14)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize = 14)
    if xmin or xmax:
        ax.set_xlim(xmin, xmax)
    if ymin or ymax:
        ax.set_ylim(ymin, ymax)
    if title:
        ax.set_title(title, fontsize = 16)
    if legend:
        ax.legend()
    ax.grid()
    ax.tick_params(axis='x', labelsize=12)
    ax.tick_params(axis='y', labelsize=12)

In [20]:
# Bulk Profile Plots Custom Profiles
i = 1
while i <= p_index_data.profile_numbers.max():

    j = 1
    p_temp = mr.MesaData(logs_path+f'/profile{i}.data')
    path_radius = 'C:/Users/abhav/Documents/UIUC_Graduate/Research/CE_GW/Temp_cluster_scp/'+str(i)+'/radius/'
    path_mass = 'C:/Users/abhav/Documents/UIUC_Graduate/Research/CE_GW/Temp_cluster_scp/'+str(i)+'/mass/'
    os.mkdir(path_radius)
    os.mkdir(path_mass)

    for name in p_temp.bulk_names:

        fig, axs = plt.subplots(1,1, figsize = (10,10))
        plot_xy(p_temp.data('logR'), p_temp.data(name), axs, xlabel = r'$log(R/R_\odot$)', ylabel = name, title = name+' vs Radius')
        plt.savefig(path_radius+str(j)+'_'+name+'_vs_Radius.png')
        plt.close()

        fig, axs = plt.subplots(1,1, figsize = (10,10))
        plot_xy(p_temp.data('mass'), p_temp.data(name), axs, xlabel = 'Mass (Msun)', ylabel = name, title = name+' vs Mass')
        plt.savefig(path_mass+str(j)+'_'+name+'_vs_Mass.png')
        plt.close()

        j+=1
        
        i += 50


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Users/abhav/Documents/UIUC_Graduate/Research/CE_GW/Temp_cluster_scp/1/radius/'

# Bulk Profile Plots
i = 1
for i in profile_array_for_age:

    j = 1
    p_temp = mr.MesaData(logs_path+f'profile{i}.data')
    path_radius = 'images_with_comp_BC/profile'+str(i)+'/radius/'
    path_mass = 'images_with_comp_BC/profile'+str(i)+'/mass/'
    # os.mkdir(path_radius)
    # os.mkdir(path_mass)

    for name in p_temp.bulk_names:

        fig, axs = plt.subplots(1,1, figsize = (10,10))
        plot_xy(p_temp.data('logR'), p_temp.data(name), axs, xlabel = r'$log(R/R_\odot$)', ylabel = name, title = name+' vs Radius')
        plt.savefig(path_radius+str(j)+'_'+name+'_vs_Radius.png')
        plt.close()

        fig, axs = plt.subplots(1,1, figsize = (10,10))
        plot_xy(p_temp.data('mass'), p_temp.data(name), axs, xlabel = 'Mass (Msun)', ylabel = name, title = name+' vs Mass')
        plt.savefig(path_mass+str(j)+'_'+name+'_vs_Mass.png')
        plt.close()

        j+=1


i = 1
p_radius = np.empty([p_index_data.profile_numbers.max(), 100])
p_age = np.empty([p_index_data.profile_numbers.max()])
while i <= p_index_data.profile_numbers.max():
    p_temp = mr.MesaData(logs_path+f'/profile{i}.data')
    p_age[i-1] = p_temp.star_age
    p_radius[i-1] = p_temp.data('logR')[0:100]
    i += 1


print(p_radius[0])

fig, axs = plt.subplots(1,1, figsize = (10,10))
for i in range(0,100):
    plot_xy(p_age, p_radius[:,i], axs, xlabel = 'Age (yr)', ylabel = r'$log(R/R_\odot$)', title = r'$log(R/R_\odot$) vs Age', color = 'C'+str(i))
plt.savefig('images_cluster/radius_vs_age.png')
plt.close()

In [22]:
# Bulk Profile Plots Combined

#custom
# profile_array_for_age = np.arange(1, 1911, 100)
p1 = mr.MesaData(logs_path+'/profile1.data')

j = 1
# bla = ['pressure_scale_height']
ymin = None
ymax = None
xmin = None
xmax = None
scatter = None
for name in p1.bulk_names:
    # p_temp = mr.MesaData(logs_path+f'/profile{profile_array_for_age[0]}.data')

    fig, axs = plt.subplots(1,1, figsize = (10,10))
    # plot_xy(10**p_temp.data('logR'), p_temp.data(name), axs, xlabel = r'$log(R/R_\odot$)', ylabel = name, title = name+' vs Radius', legend = f'Age = {round(p_temp.star_age, 5)}', ymin = ymin, ymax = ymax, xmin = xmin, xmax = xmax, scatter = scatter)

    fig2, axs2 = plt.subplots(1,1, figsize = (10,10))
    # plot_xy(p_temp.data('mass'), p_temp.data(name), axs2, xlabel = 'Mass (Msun)', ylabel = name, title = name+' vs Mass', legend = f'Age = {round(p_temp.star_age, 5)}', ymin = ymin, ymax = ymax, xmin = xmin, xmax = xmax, scatter = scatter)

    i = 1500
    while i <= p_index_data.profile_numbers.max():
        color = 1/17

        p_temp = mr.MesaData(logs_path+f'/profile{i}.data')
        plot_xy(10**p_temp.data('logR'), p_temp.data(name), axs, xlabel = r'$log(R/R_\odot$)', ylabel = name, title = name+' vs Radius', legend = f'Age = {round(p_temp.star_age, 5)}', ymin = ymin, ymax = ymax, xmin = xmin, xmax = xmax, scatter = scatter)
        plot_xy(p_temp.data('mass'), p_temp.data(name), axs2, xlabel = 'Mass (Msun)', ylabel = name, title = name+' vs Mass', legend = f'Age = {round(p_temp.star_age, 5)}', ymin = ymin, ymax = ymax, xmin = xmin, xmax = xmax, scatter = scatter)

        i += 100

    # fig.savefig('images_cluster/profiles/radius/'+str(j)+'_'+name+'_vs_Radius.png')
    # fig2.savefig('images_cluster/profiles/mass/'+str(j)+'_'+name+'_vs_Mass.png')

    j+=1
    plt.show()
    plt.close()

C:\Users\abhav\AppData\Local\Temp\ipykernel_31140\3435058312.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Comparison History Plots
fig, axs = plt.subplots(1,1,figsize=(10,10))

plot_xy(h_data.data('star_age'), 10**h_data.data('log_R'), axs, xlabel='Age (yr)')#, ylog=True)
# plot_xy(h_data.data('star_age')[1:], h_data.data('u_k')[1:], axs, xlabel='Age (yr)')#, ylog=True)
plt.show()

#Bulk History Plots
names = open('history_main_names.txt', 'r')
fig, axs = plt.subplots(1,1,figsize=(10,10))

#, xmin = 33.869534751, xmax = 33.8695347525,
i = 1
for name in names:
    plot_xy(h_data.data('star_age'), h_data.data(name.strip()), axs, xlabel='Age (yr)', ylabel=name.strip(), title=f'{name.strip()} vs Age')
    plt.savefig('images/history/'+str(i)+f'_{name.strip()}_vs_age.png')
    axs.clear()
    i+=1
plt.close()

In [14]:
name = 'velocity'
fig, axs = plt.subplots(1,1, figsize = (10,10))
# fig2, axs2 = plt.subplots(1,1, figsize = (10,10))
i = 1500
while i <= p_index_data.profile_numbers.max():

    p_temp = mr.MesaData(logs_path+f'/profile{i}.data')
    v_esc = np.sqrt(2*G_const*p_temp.data('mass')*M_sun/((10**p_temp.data('logR'))*R_sun))
    # plot_xy(10**p_temp.data('logR'), p_temp.data(name), axs, xlabel = r'$log(R/R_\odot$)', ylabel = name, title = name+' vs Radius', legend = f'Age = {round(p_temp.star_age, 5)}')
    plot_xy(p_temp.data('logR'), p_temp.data(name), axs, xlabel = 'Mass (Msun)', ylabel = name, title = name+' vs Radius', legend = f'Age = {round(p_temp.star_age, 5)}', linestyle = '-')#, ymin=0.0, ymax=0.0001)
    plot_xy(p_temp.data('logR'), v_esc, axs, xlabel = r'$log(R/R_\odot$)', ylabel = 'v_esc', title = 'v_esc(dotted) and velocity (in cm/s) vs Radius', legend = f'Age = {round(p_temp.star_age, 5)}', linestyle = 'dotted')#, ymin=0.0, ymax=0.0001)
    i += 100
fig.savefig('images_cluster/profiles/radius/84_'+name+'_vs_Radius.png')


#Individual History Plots
fig, axs = plt.subplots(1,1,figsize=(10,10))
plot_xy(h_data.data('star_age'), h_data.data('Qtb'), axs, xlabel='Age (yr)', ylabel='Qtb', title='Qtb vs Age', ylog = True)
plt.savefig('images/56_'+f'Qtb_vs_age.png')

